# Rubik's Cube Transformer: InfoNCE Contrastive Training on 2x NVIDIA T4 GPUs

This notebook trains a **Rubik's Cube Transformer Encoder** using large-scale **InfoNCE contrastive learning** on **2x NVIDIA T4 GPUs** using PyTorch Distributed Data Parallel (`torchrun` DDP).

### Topology & Contrastive Rules:
- **Positive Pair**: Exactly 1 move ahead ($\text{dist} = 1$).
- **Excluded Buffer Zone**: States with $2 \le \text{dist} \le 5$ are strictly excluded ($k=5$).
- **Negative Pairs**: Sampled states strictly with $\text{dist} > 5$ ($K=100$ negatives per anchor).
- **Streaming Generator**: Infinite on-the-fly random generation directly in RAM (zero dataset files on disk, no epoch cycling).
- **Hardware**: Dual NVIDIA T4 GPUs with Automatic Mixed Precision (AMP FP16) and parallel CPU DataLoader workers.


## 1. Hardware & Multi-GPU Detection
Verify that Kaggle has allocated **2x NVIDIA T4 GPUs** (`cuda:0` and `cuda:1`).

In [ ]:
!nvidia-smi

import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"GPU Count:       {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")

if device_count < 2:
    print("\n⚠️ Notice: Less than 2 GPUs detected. In Kaggle Notebook Settings, select 'Accelerator' -> 'GPU T4 x 2'.")
else:
    print("\n✅ Dual NVIDIA T4 GPUs successfully detected and ready for DDP training!")

## 2. Setup Repository & Dependencies
Clone the `rubik-ml-solver` repository from GitHub and install it in editable mode.

In [ ]:
import os

# Clone repository if not already present
if not os.path.exists("rubik"):
    !git clone https://github.com/luisportillo91/rubik-ml-solver.git
    %cd rubik-ml-solver
else:
    print("Working inside rubik-ml-solver repository.")

# Install package dependencies
!pip install -q -e .
!pip install -q matplotlib pytest

## 3. Verify Codebase via Test Suite
Execute the complete unit test suite to verify cube permutation logic, bidirectional BFS search, transformer encoder, and InfoNCE loss.

In [ ]:
!pytest -v tests/

## 4. Interactive InfoNCE Training (Native Python with Enriched Progress Bar)

We launch `rubik.train_infonce(...)` directly inside the notebook cell.
- **Enriched Progress Bar**: Displays real-time Loss, Top-1/Top-5 retrieval accuracy, Positive/Negative cosine similarity, and Margin.
- **Dual T4 Acceleration**: Automatically uses all available GPUs via PyTorch `DataParallel`.
- **Vectorized M x N Matrix**: M=32 anchors x N=64 canonical negative sequences = 2,048 negatives per step.
- **AMP FP16**: Uses NVIDIA Turing Tensor Cores for maximum throughput.
- **Checkpoints**: Automatically saves the best model to `/kaggle/working/checkpoints/best_encoder.pt`.

In [ ]:
import rubik

# Run training directly in the notebook with live enriched progress bar
result = rubik.train_infonce(
    steps=2000,
    batch_size=32,          # M random anchors
    k_negatives=64,         # N canonical negatives (M x N = 2,048 negatives per step)
    distance_threshold=5,   # Excluded buffer zone [2, 5]
    d_model=128,
    nhead=4,
    num_layers=4,
    d_embed=128,
    lr=3e-4,
    device="auto",          # Automatically uses 2x T4 GPUs via DataParallel
    multi_gpu=True,
    amp=True,
    log_every=25,
    save_every=250,
    save_dir="/kaggle/working/checkpoints",
    history_file="training_history.json",
    show_progress_bar=True,
)

print(f"\nTraining finished! Best Loss: {result.best_loss:.4f} at Step {result.best_step}")


## 5. Visualize Training Curves
Plot the evolution of InfoNCE loss, Top-1/Top-5 accuracy, positive vs negative cosine similarity, and similarity margin directly from .

In [ ]:
# Plot directly using the returned result object
fig = result.plot_curves(save_path="/kaggle/working/training_curves.png")


## 6. Learned Distance Manifold Evaluation
Verify whether the trained transformer encoder has learned a monotonic distance metric on the Rubik's cube graph.
We measure cosine similarity between cube states as a function of scramble distance $d \in [1, 20]$.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from rubik.cube import create_solved_cube, scramble
from rubik.encoder import CubeTransformerEncoder, batch_to_tensor

best_checkpoint = os.path.join(save_dir, "best_encoder.pt")
ckpt = torch.load(best_checkpoint, map_location="cpu")
saved_args = ckpt.get("config", ckpt.get("args", {}))

eval_model = CubeTransformerEncoder(
    d_model=saved_args.get("d_model", 128),
    nhead=saved_args.get("nhead", 4),
    num_layers=saved_args.get("num_layers", 4),
    dim_feedforward=saved_args.get("d_model", 128) * 2,
    d_embed=saved_args.get("d_embed", 128),
    pooling="cls",
)
eval_model.load_state_dict(ckpt["model_state_dict"])
eval_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_model.to(device)

distances = list(range(1, 21))
trials_per_distance = 64
mean_sims = []
std_sims = []

solved = create_solved_cube()

with torch.no_grad():
    for d in distances:
        sims = []
        for _ in range(trials_per_distance):
            s1, _ = scramble(solved, n_moves=12)
            s2, _ = scramble(s1, n_moves=d)
            
            t1 = batch_to_tensor([s1], device=device)
            t2 = batch_to_tensor([s2], device=device)
            
            e1, _ = eval_model(t1)
            e2, _ = eval_model(t2)
            
            e1_norm = F.normalize(e1, p=2, dim=-1)
            e2_norm = F.normalize(e2, p=2, dim=-1)
            sim = (e1_norm * e2_norm).sum(dim=-1).item()
            sims.append(sim)
        mean_sims.append(float(np.mean(sims)))
        std_sims.append(float(np.std(sims)))

plt.figure(figsize=(10, 5))
plt.plot(distances, mean_sims, marker='o', color='#1f77b4', lw=2, label='Average Cosine Similarity')
plt.fill_between(
    distances,
    [m - s for m, s in zip(mean_sims, std_sims)],
    [m + s for m, s in zip(mean_sims, std_sims)],
    alpha=0.2,
    color='#1f77b4',
)
plt.axvline(x=1, color='green', linestyle='--', label='Positive (dist=1)')
plt.axvspan(2, 5, color='orange', alpha=0.2, label='Excluded Zone (k=5)')
plt.axvline(x=5.5, color='red', linestyle='--', label='Negative Zone (dist>5)')
plt.title("Learned Representation Similarity vs Scramble Distance", fontsize=14, fontweight="bold")
plt.xlabel("Scramble Distance (moves)", fontsize=12)
plt.ylabel("Cosine Similarity", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
manifold_plot_path = "/kaggle/working/similarity_vs_distance.png"
plt.savefig(manifold_plot_path, dpi=150)
print(f"Distance manifold plot saved to: {manifold_plot_path}")
plt.show()

## 7. Output Artifacts
List all saved model checkpoints and visualization plots in `/kaggle/working/` ready for download from the Kaggle sidebar.

In [ ]:
!ls -lh checkpoints/
!ls -lh *.png

# 1. Interactive download link directly in the notebook
from IPython.display import FileLink, display
print("Click below to download best_model.pt directly:")
display(FileLink("checkpoints/best_model.pt"))

# 2. Bundle all checkpoints and plots into a single zip
!zip -q -r rubik_artifacts.zip checkpoints/ *.png
print("
Click below to download all checkpoints and plots as a zip archive:")
display(FileLink("rubik_artifacts.zip"))

print("
🎉 Training and evaluation finished successfully!")